In [ ]:
%pip install qiskit matplotlib qiskit[visualization]
%pip install qiskit-aer

In [21]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

qc = QuantumCircuit(17) # input: 8 (0~7번), ancilla: 1 (8번), output: 1 (9번)
# 10-17번: diffuser용 ancilla
simulator = AerSimulator()

def initial():
    for i in range(9):
        qc.h(i)

def oracle():
    qc.cx(5,0)
    qc.cx(6,1)
    qc.cx(7,2)
    qc.cx(4,3)

    qc.barrier()

    # x 게이트 통과
    qc.x(0)
    qc.x(1)
    qc.x(2)

    # cccz 게이트 구현
    qc.ccx(0,1,8)
    qc.ccz(2,8,9)

def de_oracle():
    # q(8) 원복
    qc.ccx(0,1,8)

    # x 게이트 원복
    qc.x(0)
    qc.x(1)
    qc.x(2)

    qc.barrier()

    qc.cx(4,3)
    qc.cx(7,2)
    qc.cx(6,1)
    qc.cx(5,0)

def diffuser():
    qc.barrier()
    for i in range(7):
        qc.h(i)
    for i in range(7):
        qc.x(i)
    qc.z(7)

    qc.barrier()

    # cccc..cx 게이트 구현
    qc.ccx(0,1,10)
    qc.ccx(2,3,11)
    qc.ccx(4,5,12)
    qc.ccx(6,10,13)
    qc.ccx(11,12,14)
    qc.ccx(13,14,7)
    
    # 원복
    qc.ccx(11,12,14)
    qc.ccx(6,10,13)
    qc.ccx(4,5,12)
    qc.ccx(2,3,11)
    qc.ccx(0,1,10)

    qc.barrier()
        
    for i in range(7):
        qc.x(i)
    for i in range(7):
        qc.h(i)
    qc.z(7)

initial()
for i in range(2):
    oracle()
    de_oracle()
    diffuser()
    qc.barrier()

qc.measure_all()

qc.draw(output='mpl', filename='grover.png')
backend = AerSimulator()
job = backend.run(transpile(qc, backend))
result = job.result()

counts = result.get_counts(qc)
print("AER counts: ", counts)
fig = plot_histogram(counts)
fig.savefig('grover_histogram.png')


AER counts:  {'00000000000110100': 4, '00000000000000011': 4, '00000000101100000': 3, '00000000110001011': 2, '00000000000011111': 2, '00000000101010001': 2, '00000000001100011': 4, '00000000100100000': 2, '00000000010001110': 4, '00000000001010000': 2, '00000000010111110': 3, '00000000101110001': 3, '00000000100001100': 4, '00000000001001101': 2, '00000000001110001': 3, '00000000100010000': 1, '00000000111111110': 5, '00000000101100101': 3, '00000000111001111': 3, '00000000100101010': 1, '00000000001101011': 3, '00000000011010111': 1, '00000000001110010': 3, '00000000100010011': 3, '00000000111100100': 2, '00000000111000110': 4, '00000000001100010': 3, '00000000100100001': 4, '00000000110110010': 1, '00000000110010110': 2, '00000000100101100': 4, '00000000111011100': 1, '00000000110110001': 1, '00000000010100111': 2, '00000000101001100': 3, '00000000000011101': 2, '00000000100111110': 4, '00000000101010011': 3, '00000000011010110': 1, '00000000100011111': 3, '00000000001000010': 2, '0